In [0]:
from pyspark.sql import functions as F

TABLAS = [
    "campaign_desc", "campaign_table", "coupon", "coupon_redempt",
    "hh_demographic", "product", "transaction_data",
]

filas = []
for nombre in TABLAS:
    df = spark.table(f"dunnhumby.bronze.{nombre}")
    total = df.count()
    columnas = [c for c in df.columns if not c.startswith("_")]  # se omiten las columnas de auditoría

    # Bronze guarda todo como texto, así que un valor vacío ("") también cuenta como faltante
    conteos = df.select([
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in columnas
    ]).first()

    for c in columnas:
        filas.append((nombre, c, total, conteos[c], round(100 * conteos[c] / total, 2)))

resultado = spark.createDataFrame(
    filas, ["tabla", "columna", "total_filas", "nulos_o_vacios", "pct_faltante"]
)
display(resultado.orderBy(F.desc("pct_faltante")))

In [0]:
# Clave natural de cada tabla: la combinación de columnas que debería ser única
CLAVES = {
    "campaign_desc":    ["CAMPAIGN"],
    "campaign_table":   ["household_key", "CAMPAIGN"],
    "coupon":           ["COUPON_UPC", "CAMPAIGN", "PRODUCT_ID"],
    "coupon_redempt":   ["household_key", "DAY", "COUPON_UPC", "CAMPAIGN"],
    "hh_demographic":   ["household_key"],
    "product":          ["PRODUCT_ID"],
    "transaction_data": ["BASKET_ID", "PRODUCT_ID"],
}

filas = []
for nombre, clave in CLAVES.items():
    df = spark.table(f"dunnhumby.bronze.{nombre}")
    total = df.count()
    columnas = [c for c in df.columns if not c.startswith("_")]  # se omiten las columnas de auditoría
    unicas_clave = df.select(clave).distinct().count()
    unicas_fila = df.select(columnas).distinct().count()
    filas.append((nombre, total, total - unicas_clave, total - unicas_fila))

resultado = spark.createDataFrame(
    filas, ["tabla", "total_filas", "repetidas_por_clave", "repetidas_filas_identicas"]
)
display(resultado)

In [0]:
display(
    spark.table("dunnhumby.bronze.product")
    .filter("DEPARTMENT IS NULL OR TRIM(DEPARTMENT) = ''")
    .drop("_archivo_origen", "_fecha_carga")
)

In [0]:
sin_clasificar = (
    spark.table("dunnhumby.bronze.product")
    .filter("DEPARTMENT IS NULL OR TRIM(DEPARTMENT) = ''")
    .select("PRODUCT_ID")
)

tx = spark.table("dunnhumby.bronze.transaction_data")
tx_sc = tx.join(sin_clasificar, "PRODUCT_ID", "inner")

# En Bronze todo es texto, por eso SALES_VALUE se convierte a número para poder sumarlo
ventas_sc = tx_sc.agg(F.sum(F.col("SALES_VALUE").cast("double"))).first()[0] or 0
ventas_total = tx.agg(F.sum(F.col("SALES_VALUE").cast("double"))).first()[0]
cupones_sc = spark.table("dunnhumby.bronze.coupon").join(sin_clasificar, "PRODUCT_ID", "inner").count()

print(f"Líneas de transacción con esos productos: {tx_sc.count():,}")
print(f"Ventas de esos productos: {ventas_sc:,.2f} ({100 * ventas_sc / ventas_total:.4f}% del total)")
print(f"Filas de coupon con esos productos: {cupones_sc:,}")

In [0]:
from functools import reduce

def valores_distintos(tabla, columnas):
    """Una fila por cada valor distinto de cada columna, con su cantidad de filas."""
    df = spark.table(f"dunnhumby.bronze.{tabla}")
    partes = [df.select(F.lit(c).alias("columna"), F.col(c).alias("valor")) for c in columnas]
    largo = reduce(lambda a, b: a.unionByName(b), partes)
    return largo.groupBy("columna", "valor").count().orderBy("columna", "valor")

display(valores_distintos("hh_demographic", [
    "AGE_DESC", "MARITAL_STATUS_CODE", "INCOME_DESC", "HOMEOWNER_DESC",
    "HH_COMP_DESC", "HOUSEHOLD_SIZE_DESC", "KID_CATEGORY_DESC",
]))
display(valores_distintos("product", ["DEPARTMENT", "BRAND"]))

In [0]:
from pyspark.sql import functions as F   # se importa aquí porque la sesión puede reiniciarse entre celdas

COLUMNAS_NUM = ["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "COUPON_DISC",
                "COUPON_MATCH_DISC", "DAY", "TRANS_TIME", "WEEK_NO"]
tx = spark.table("dunnhumby.bronze.transaction_data")

filas = []
for c in COLUMNAS_NUM:
    # try_cast devuelve NULL si el texto no es un número, en lugar de lanzar error
    n = F.expr(f"try_cast(`{c}` AS DOUBLE)")
    r = tx.agg(
        F.min(n).alias("minimo"),
        F.max(n).alias("maximo"),
        F.sum(F.when(n == 0, 1).otherwise(0)).alias("ceros"),
        F.sum(F.when(n < 0, 1).otherwise(0)).alias("negativos"),
        F.sum(F.when(n.isNull(), 1).otherwise(0)).alias("no_numericos"),
    ).first()
    filas.append((c, r["minimo"], r["maximo"], r["ceros"], r["negativos"], r["no_numericos"]))

display(spark.createDataFrame(
    filas, ["columna", "minimo", "maximo", "ceros", "negativos", "no_numericos"]
))

In [0]:
sin_clasificar = (
    spark.table("dunnhumby.bronze.product")
    .filter("DEPARTMENT IS NULL OR TRIM(DEPARTMENT) = ''")
    .select("PRODUCT_ID")
)
tx_sc = tx.join(sin_clasificar, "PRODUCT_ID", "inner")
display(tx_sc.groupBy("QUANTITY", "SALES_VALUE").count().orderBy(F.desc("count")))

In [0]:
hogares_tx = tx.select("household_key").distinct()
hogares_dem = spark.table("dunnhumby.bronze.hh_demographic").select("household_key").distinct()

n_tx = hogares_tx.count()
n_con_dem = hogares_tx.join(hogares_dem, "household_key", "inner").count()

print(f"Hogares con compras: {n_tx:,}")
print(f"Hogares con compras y datos demográficos: {n_con_dem:,} ({100 * n_con_dem / n_tx:.1f}%)")

In [0]:
from pyspark.sql import functions as F

tx = spark.table("dunnhumby.bronze.transaction_data")
q = F.expr("try_cast(QUANTITY AS DOUBLE)")
s = F.expr("try_cast(SALES_VALUE AS DOUBLE)")

# Las cuatro combinaciones posibles: nos dicen si "cantidad cero" y "venta cero" coinciden
display(
    tx.groupBy((q == 0).alias("cantidad_cero"), (s == 0).alias("venta_cero"))
      .count()
      .orderBy("cantidad_cero", "venta_cero")
)

In [0]:
display(
    tx.select("household_key", "BASKET_ID", "PRODUCT_ID", "QUANTITY", "SALES_VALUE", "RETAIL_DISC")
      .withColumn("cantidad_num", q)
      .orderBy(F.desc("cantidad_num"))
      .drop("cantidad_num")
      .limit(10)
)

In [0]:
rd = F.expr("try_cast(RETAIL_DISC AS DOUBLE)")
positivos = tx.filter(rd > 0)
print(f"Líneas con RETAIL_DISC positivo: {positivos.count():,}")
display(positivos.select("BASKET_ID", "PRODUCT_ID", "QUANTITY", "SALES_VALUE", "RETAIL_DISC").limit(10))

In [0]:
from pyspark.sql import functions as F

tx = spark.table("dunnhumby.bronze.transaction_data")
pr = spark.table("dunnhumby.bronze.product")
q = F.expr("try_cast(QUANTITY AS DOUBLE)")

display(
    pr.filter("PRODUCT_ID IN ('6534178', '6544236')")
      .drop("_archivo_origen", "_fecha_carga")
)

In [0]:
# 1,000 es un umbral arbitrario, muy por encima de una compra normal
grandes = (
    tx.filter(q > 1000)
      .join(pr.select("PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"), "PRODUCT_ID", "left")
      .groupBy("PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC").count()
      .orderBy(F.desc("count"))
)
display(grandes)

In [0]:
display(
    pr.filter("DEPARTMENT IN ('MISC SALES TRAN', 'MISC. TRANS.')")
      .groupBy("DEPARTMENT", "COMMODITY_DESC").count()
      .orderBy("DEPARTMENT", "COMMODITY_DESC")
)